# 75 — Initial National Report Builder

Builds a PDF with tables, graphs, conclusions, news and appendix.

In [ ]:
import os, io, re, json, hashlib, platform, sys, zipfile
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def safe_read_parquet(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        import pyarrow.parquet as pq
        df = pq.read_table(p).to_pandas()
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def clean_facility(fn: str) -> str:
    s = re.sub(r"\.xlsm$|\.xlsx$|\.pdf$", "", str(fn), flags=re.I)
    s = re.sub(r"^[A-Z0-9]+\s+", "", s)
    s = re.sub(r"\s+Annual Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Annual Performance Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Report 2024$", "", s, flags=re.I)
    s = re.sub(r"\s+2024$", "", s)
    return s.strip()

run_cfg = load_yaml(CONFIGS / "run.yml")
phase70_cfg = load_yaml(CONFIGS / "phase70.yml")
doc_cfg = load_yaml(CONFIGS / "documentary_sources.yml")

In [ ]:
step = "75_initial_national_report_builder"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

targets, _ = safe_read_csv(OUTPUTS / "72_target_control_registry_builder" / "target_control_registry.csv")
news_context, _ = safe_read_csv(OUTPUTS / "73_news_satellite_context_pack" / "news_context_for_report.csv")
site_summary, _ = safe_read_csv(OUTPUTS / "42_multi_site_comparison" / "site_summary.csv")
integrity = load_json(OUTPUTS / "60_final_integrity_report" / "final_integrity_report.json") if (OUTPUTS / "60_final_integrity_report" / "final_integrity_report.json").exists() else {}
doc_summary = load_json(OUTPUTS / "61_report_catalog_ingest" / "documentary_ingest_summary.json") if (OUTPUTS / "61_report_catalog_ingest" / "documentary_ingest_summary.json").exists() else {}

if targets.empty:
    raise FileNotFoundError("Need outputs/72_target_control_registry_builder/target_control_registry.csv")

chart1 = out / "top_targets.png"
plt.figure(figsize=(10,5))
plot_df = targets.head(10).copy()
plt.barh(plot_df["facility"][::-1], plot_df["integrated_screening_score"][::-1])
plt.xlabel("Integrated screening score")
plt.title("Top target shortlist")
plt.tight_layout()
plt.savefig(chart1, dpi=200)
plt.close()

chart2 = out / "site_scores.png"
if not site_summary.empty:
    plt.figure(figsize=(8,4.5))
    plt.bar(site_summary["site_name"], site_summary["mean_site_score"])
    plt.title("Current target/control site scores")
    plt.ylabel("Mean site score")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(chart2, dpi=200)
    plt.close()

pdf_path = out / "AQ26_phase70_initial_report.pdf"
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="TitleBig", parent=styles["Title"], fontSize=20, leading=24, alignment=TA_CENTER, spaceAfter=8))
styles.add(ParagraphStyle(name="Section", parent=styles["Heading1"], fontSize=14, leading=18, spaceBefore=10, spaceAfter=6))
styles.add(ParagraphStyle(name="Body", parent=styles["BodyText"], fontSize=10.5, leading=14))
styles.add(ParagraphStyle(name="Subtle", parent=styles["BodyText"], fontSize=9, textColor=colors.grey))
doc = SimpleDocTemplate(str(pdf_path), pagesize=A4, rightMargin=16*mm, leftMargin=16*mm, topMargin=14*mm, bottomMargin=14*mm)
story = []
story.append(Paragraph("AQ26 Phase 70 Initial National Screening Report", styles["TitleBig"]))
story.append(Paragraph("Initial integrated screening report using documentary evidence, current environmental pipeline outputs, news context, and configured control location.", styles["Body"]))
story.append(Paragraph("This is a prioritization and integrity-driven screening report, not a final causal attribution report.", styles["Subtle"]))
story.append(Spacer(1,6))
story.append(Paragraph("1. Executive summary", styles["Section"]))
story.append(Paragraph(
    f"The current repository integrates documentary emissions evidence, ground-source, weather, news, target-control comparison, contradiction logging, and an AI reporting layer. "
    f"The strongest current screening signal comes from the documentary corpus. The configured control location is {targets['control_location'].iloc[0]}. "
    f"The full forensic run still reports limitations, including missing dependencies count={integrity.get('missing_dependency_count', 'n/a')} and contradiction count={integrity.get('contradiction_count', 'n/a')}. "
    f"The documentary corpus currently covers {doc_summary.get('catalog_rows', 'n/a')} reports and {doc_summary.get('emissions_rows', 'n/a')} normalized emissions rows.",
    styles["Body"]
))
story.append(Paragraph("2. Target shortlist", styles["Section"]))
story.append(Image(str(chart1), width=160*mm, height=82*mm))
rows = [["Rank","Permit","Facility","Risk","Control"]]
for _, r in targets.head(10).iterrows():
    rows.append([int(r["target_rank"]), str(r["permit_id"]), str(r["facility"])[:35], f"{float(r['integrated_screening_score']):.1f}", str(r["control_location"])])
tbl = Table(rows, colWidths=[12*mm,22*mm,78*mm,22*mm,42*mm])
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#163a63")),
    ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 8),
    ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.lightgrey]),
]))
story.append(tbl)
story.append(Spacer(1,8))
if chart2.exists():
    story.append(Paragraph("3. Current configured target/control evidence", styles["Section"]))
    story.append(Image(str(chart2), width=150*mm, height=80*mm))
story.append(Paragraph("4. News context", styles["Section"]))
news_rows = [["Published","Headline"]]
for _, r in news_context.head(8).iterrows():
    news_rows.append([str(r.get("published_at",""))[:10], str(r.get("title",""))[:100]])
news_tbl = Table(news_rows, colWidths=[24*mm,145*mm])
news_tbl.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#245b8f")),
    ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 8.5),
    ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.lightgrey]),
]))
story.append(news_tbl)
story.append(Spacer(1,8))
story.append(Paragraph("5. Conclusions", styles["Section"]))
for line in [
    "The system is now capable of producing integrated national screening outputs.",
    "The documentary layer is currently the strongest source of prioritization power.",
    "Ground, weather, satellite, and AI layers are present but not all are yet equally mature or equally evidential.",
    "The shortlist should be used to drive the next cycle of deeper targeted review.",
]:
    story.append(Paragraph(f"- {line}", styles["Body"]))
story.append(PageBreak())
story.append(Paragraph("Appendix A. Integrity snapshot", styles["Section"]))
app = [["Metric","Value"],
       ["Missing dependency count", str(integrity.get("missing_dependency_count", "n/a"))],
       ["Contradiction count", str(integrity.get("contradiction_count", "n/a"))],
       ["Repo ready for forensic review", str(integrity.get("repo_ready_for_forensic_review", "n/a"))],
       ["Documentary reports", str(doc_summary.get("catalog_rows", "n/a"))],
       ["Documentary emissions rows", str(doc_summary.get("emissions_rows", "n/a"))]]
app_tbl = Table(app, colWidths=[90*mm,60*mm])
app_tbl.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#163a63")),
    ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.lightgrey]),
]))
story.append(app_tbl)
doc.build(story)

manifest = manifest_base(step, [CONFIGS / "phase70.yml"])
add_artifact(manifest, pdf_path)
add_artifact(manifest, chart1)
if chart2.exists(): add_artifact(manifest, chart2)
write_json(out / "manifest.json", manifest)
print(str(pdf_path))